# Hybrid Quantum + RNN TSP Solver

This notebook implements a **hybrid quantum-classical model** for the Traveling Salesman Problem (TSP):

- a **PyTorch RNN decoder** predicts a tour sequentially,
- a **Qiskit quantum neural network** scores candidate city-to-city transitions,
- the **difficulty** can be adjusted by changing `num_cities`.

For small TSP sizes, labels are generated by **exact brute force**.
For larger sizes, labels switch to **nearest-neighbor + 2-opt** heuristics so the dataset still scales.

In [ ]:
# Install dependencies if needed
!pip install torch numpy qiskit qiskit-machine-learning matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.9 MB/s eta 0:00:00


In [ ]:
# Needed for hardware running
!pip install qiskit_ibm_runtime

In [32]:
import itertools
import math
import random
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN

In [33]:
# Hardware imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

## Configuration

Change `num_cities` to make the problem easier or harder.
- `num_cities <= exact_threshold`: exact labels via brute force
- `num_cities > exact_threshold`: heuristic labels via nearest-neighbor + 2-opt

In [34]:
CONFIG = {
    "num_cities": 7,          # main difficulty knob
    "train_size": 256,
    "val_size": 64,
    "batch_size": 16,
    "hidden_dim": 128,
    "num_qubits": 4,
    "epochs": 8,
    "lr": 2e-3,
    "exact_threshold": 9,
    "seed": 42,
}

In [35]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### Connect to RPI-IBM 127-qubit Quantum System One backend

RPI-IBM service creds via service.ipynb

In [36]:
service = QiskitRuntimeService(instance="crn:v1:bluemix:public:quantum-computing:us-east:a/b8ff6077c08a4ea9871560ccb827d457:d3452110-b228-4c79-8959-15ea8cfd435d::")
backend = service.backend("ibm_rensselaer")

## TSP utilities

In [37]:
def pairwise_dist(points: np.ndarray) -> np.ndarray:
    diff = points[:, None, :] - points[None, :, :]
    return np.sqrt((diff**2).sum(axis=-1))


def tour_length(points: np.ndarray, tour: List[int]) -> float:
    dist = pairwise_dist(points)
    total = 0.0
    for i in range(len(tour)):
        total += dist[tour[i], tour[(i + 1) % len(tour)]]
    return float(total)


def brute_force_tour(points: np.ndarray, start_city: int = 0) -> List[int]:
    n = len(points)
    others = [i for i in range(n) if i != start_city]
    best_len = float("inf")
    best_tour = None
    for perm in itertools.permutations(others):
        tour = [start_city, *perm]
        cur_len = tour_length(points, tour)
        if cur_len < best_len:
            best_len = cur_len
            best_tour = tour
    assert best_tour is not None
    return best_tour


def nearest_neighbor_tour(points: np.ndarray, start_city: int = 0) -> List[int]:
    n = len(points)
    dist = pairwise_dist(points)
    unvisited = set(range(n))
    unvisited.remove(start_city)
    tour = [start_city]
    cur = start_city
    while unvisited:
        nxt = min(unvisited, key=lambda j: dist[cur, j])
        tour.append(nxt)
        unvisited.remove(nxt)
        cur = nxt
    return tour


def two_opt(points: np.ndarray, tour: List[int]) -> List[int]:
    best = tour[:]
    best_len = tour_length(points, best)
    improved = True
    while improved:
        improved = False
        n = len(best)
        for i in range(1, n - 2):
            for j in range(i + 1, n):
                if j - i == 1:
                    continue
                candidate = best[:]
                candidate[i:j] = reversed(candidate[i:j])
                cand_len = tour_length(points, candidate)
                if cand_len + 1e-12 < best_len:
                    best = candidate
                    best_len = cand_len
                    improved = True
    return best


def build_label_tour(points: np.ndarray, exact_threshold: int = 9) -> Tuple[List[int], bool]:
    n = len(points)
    if n <= exact_threshold:
        return brute_force_tour(points), True
    heuristic = nearest_neighbor_tour(points)
    heuristic = two_opt(points, heuristic)
    return heuristic, False


def describe_difficulty(num_cities: int, exact_threshold: int) -> str:
    if num_cities <= exact_threshold:
        return (
            f"num_cities={num_cities}: labels are exact optimal tours via brute force. "
            f"This is easier for supervised learning but label generation grows factorially."
        )
    return (
        f"num_cities={num_cities}: labels use nearest-neighbor + 2-opt heuristic. "
        f"This scales better and is a harder regime."
    )

print(describe_difficulty(CONFIG["num_cities"], CONFIG["exact_threshold"]))

num_cities=7: labels are exact optimal tours via brute force. This is easier for supervised learning but label generation grows factorially.


## Dataset

In [38]:
@dataclass
class TSPExample:
    points: np.ndarray
    label_tour: List[int]
    exact: bool


class TSPDataset(Dataset):
    def __init__(self, num_samples: int, num_cities: int, exact_threshold: int = 9, seed: int = 0):
        self.examples: List[TSPExample] = []
        rng = np.random.default_rng(seed)
        for _ in range(num_samples):
            points = rng.random((num_cities, 2), dtype=np.float32)
            label_tour, exact = build_label_tour(points, exact_threshold=exact_threshold)
            self.examples.append(TSPExample(points=points, label_tour=label_tour, exact=exact))

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        ex = self.examples[idx]
        return {
            "points": torch.tensor(ex.points, dtype=torch.float32),
            "tour": torch.tensor(ex.label_tour, dtype=torch.long),
            "exact": torch.tensor(int(ex.exact), dtype=torch.long),
        }


def collate_fn(batch):
    points = torch.stack([b["points"] for b in batch], dim=0)
    tours = torch.stack([b["tour"] for b in batch], dim=0)
    exact = torch.stack([b["exact"] for b in batch], dim=0)
    return points, tours, exact

## Quantum edge scorer

In [46]:
class QuantumEdgeScorer(nn.Module):
    """
    Input edge feature: [x_i, y_i, x_j, y_j]
    Output: scalar score for transition i -> j
    """
    def __init__(self, num_qubits: int = 4, backend=None):
        super().__init__()
        self.num_qubits = num_qubits
        self.in_proj = nn.Linear(4, num_qubits)

        x = ParameterVector("x", num_qubits)
        theta = ParameterVector("theta", num_qubits)

        qc = QuantumCircuit(num_qubits)
        for i in range(num_qubits):
            qc.rx(x[i], i)
            qc.ry(theta[i], i)
        for i in range(num_qubits - 1):
            qc.cz(i, i + 1)
        if num_qubits > 2:
            qc.cz(num_qubits - 1, 0)

        observable = SparsePauliOp.from_list([("Z" * num_qubits, 1.0)])


        # Need to transpile first before running on hardware
        if backend is not None:
            pm = generate_preset_pass_manager(
                optimization_level=1,
                backend=backend
            )
            isa_circuit = pm.run(qc)
            isa_observable = observable.apply_layout(isa_circuit.layout)
        else:
            isa_circuit = qc
            isa_observable = observable

        print("ISA ops:", isa_circuit.count_ops())


        # Establish estimator connected to hardware backend
        estimator = Estimator(backend) if backend is not None else None


        qnn = EstimatorQNN(
            circuit=qc,
            input_params=list(x),
            weight_params=list(theta),
            observables=observable,
            input_gradients=True,
            # connect to quantum hardware estimator
            estimator=estimator
        )
        self.q_layer = TorchConnector(qnn)
        self.out_proj = nn.Linear(1, 1)

    def forward(self, edge_features: torch.Tensor) -> torch.Tensor:
        q_inputs = math.pi * torch.sigmoid(self.in_proj(edge_features))
        q_out = self.q_layer(q_inputs)
        if q_out.dim() == 1:
            q_out = q_out.unsqueeze(-1)
        return self.out_proj(q_out).squeeze(-1)

## Hybrid RNN + quantum TSP model

In [47]:
class HybridQuantumRNNPointer(nn.Module):
    def __init__(self, hidden_dim: int = 128, num_qubits: int = 4):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.city_embed = nn.Linear(2, hidden_dim)
        self.encoder = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.decoder_cell = nn.GRUCell(hidden_dim, hidden_dim)

        self.start_token = nn.Parameter(torch.zeros(hidden_dim))
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.logit_scale = nn.Parameter(torch.tensor(1.0))
        self.quantum_alpha = nn.Parameter(torch.tensor(0.5))

        self.q_edge_scorer = QuantumEdgeScorer(num_qubits=num_qubits, backend=backend)

    def _step_logits(self, points, enc_states, dec_hidden, current_city_idx, visited_mask):
        B, N, _ = points.shape
        H = enc_states.size(-1)

        query = self.query_proj(dec_hidden).unsqueeze(1)
        keys = self.key_proj(enc_states)
        classical_logits = (query * keys).sum(dim=-1) / math.sqrt(H)

        cur_pts = points[torch.arange(B, device=points.device), current_city_idx]
        cur_pts = cur_pts.unsqueeze(1).expand(B, N, 2)
        edge_feats = torch.cat([cur_pts, points], dim=-1).reshape(B * N, 4)
        quantum_logits = self.q_edge_scorer(edge_feats).reshape(B, N)

        logits = self.logit_scale * classical_logits + self.quantum_alpha * quantum_logits
        logits = logits.masked_fill(visited_mask, -1e9)
        return logits

    def forward(self, points: torch.Tensor, target_tour: Optional[torch.Tensor] = None, teacher_forcing: bool = True):
        B, N, _ = points.shape
        emb = self.city_embed(points)
        enc_states, enc_hidden = self.encoder(emb)
        dec_hidden = enc_hidden.squeeze(0)

        visited = torch.zeros(B, N, dtype=torch.bool, device=points.device)
        current_city = torch.zeros(B, dtype=torch.long, device=points.device)
        visited[:, 0] = True

        prev_input = self.start_token.unsqueeze(0).expand(B, -1)

        logits_steps = []
        pred_tours = [current_city]

        for step in range(N - 1):
            dec_hidden = self.decoder_cell(prev_input, dec_hidden)
            logits = self._step_logits(points, enc_states, dec_hidden, current_city, visited)
            logits_steps.append(logits)

            pred_next = logits.argmax(dim=-1)
            if teacher_forcing and target_tour is not None:
                next_city = target_tour[:, step + 1]
            else:
                next_city = pred_next

            pred_tours.append(pred_next)
            visited = visited.clone()
            visited[torch.arange(B, device=points.device), next_city] = True
            current_city = next_city
            prev_input = enc_states[torch.arange(B, device=points.device), next_city]

        logits_steps = torch.stack(logits_steps, dim=1)
        pred_tours = torch.stack(pred_tours, dim=1)
        return logits_steps, pred_tours

    @torch.no_grad()
    def greedy_decode(self, points: torch.Tensor) -> torch.Tensor:
        _, pred_tours = self.forward(points, target_tour=None, teacher_forcing=False)
        return pred_tours

## Training and evaluation

In [48]:
import tqdm

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_count = 0

    for points, tours, _ in tqdm.tqdm(loader, desc="Training"):
        points = points.to(device)
        tours = tours.to(device)

        logits_steps, _ = model(points, target_tour=tours, teacher_forcing=True)
        targets = tours[:, 1:]
        loss = F.cross_entropy(logits_steps.reshape(-1, logits_steps.size(-1)), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * points.size(0)
        total_count += points.size(0)

    return total_loss / max(total_count, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_pred_len = 0.0
    total_label_len = 0.0
    total_exact_gap = 0.0
    exact_count = 0
    count = 0

    for points, tours, exact in loader:
        points = points.to(device)
        pred_tours = model.greedy_decode(points).cpu().numpy()
        pts_np = points.cpu().numpy()
        tours_np = tours.numpy()
        exact_np = exact.numpy()

        for pts, pred, label, is_exact in zip(pts_np, pred_tours, tours_np, exact_np):
            pred_list = list(map(int, pred.tolist()))
            label_list = list(map(int, label.tolist()))
            pred_len = tour_length(pts, pred_list)
            label_len = tour_length(pts, label_list)

            total_pred_len += pred_len
            total_label_len += label_len
            if is_exact:
                total_exact_gap += pred_len - label_len
                exact_count += 1
            count += 1

    return {
        "avg_pred_tour_length": total_pred_len / max(count, 1),
        "avg_label_tour_length": total_label_len / max(count, 1),
        "avg_exact_gap": total_exact_gap / max(exact_count, 1) if exact_count > 0 else None,
        "exact_eval_count": exact_count,
    }

## Build datasets and model

In [49]:
train_ds = TSPDataset(
    num_samples=CONFIG["train_size"],
    num_cities=CONFIG["num_cities"],
    exact_threshold=CONFIG["exact_threshold"],
    seed=CONFIG["seed"],
)

val_ds = TSPDataset(
    num_samples=CONFIG["val_size"],
    num_cities=CONFIG["num_cities"],
    exact_threshold=CONFIG["exact_threshold"],
    seed=CONFIG["seed"] + 1,
)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)

model = HybridQuantumRNNPointer(
    hidden_dim=CONFIG["hidden_dim"],
    num_qubits=CONFIG["num_qubits"],
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

print(describe_difficulty(CONFIG["num_cities"], CONFIG["exact_threshold"]))
print(model.__class__.__name__)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


ISA ops: OrderedDict({'rz': 48, 'sx': 26, 'ecr': 10, 'x': 3})
num_cities=7: labels are exact optimal tours via brute force. This is easier for supervised learning but label generation grows factorially.
HybridQuantumRNNPointer


In [51]:
print(model.q_edge_scorer.q_layer._neural_network._circuit.count_ops())
# Should show rz/sx/ecr/x — no rx

OrderedDict({'rx': 4, 'ry': 4, 'cz': 4})


## Train

In [50]:
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    metrics = evaluate(model, val_loader, device)
    exact_gap_str = f"{metrics['avg_exact_gap']:.4f}" if metrics["avg_exact_gap"] is not None else "N/A"

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"pred_len={metrics['avg_pred_tour_length']:.4f} | "
        f"label_len={metrics['avg_label_tour_length']:.4f} | "
        f"exact_gap={exact_gap_str} | "
        f"exact_cases={metrics['exact_eval_count']}"
    )

Training:   0%|                   | 0/16 [00:00<?, ?it/s]


IBMInputValueError: 'The instruction rx on qubits (0,) is not supported by the target system. Circuits that do not match the target hardware definition are no longer supported after March 4, 2024. See the transpilation documentation (https://quantum.cloud.ibm.com/docs/guides/transpile) for instructions to transform circuits and the primitive examples (https://quantum.cloud.ibm.com/docs/guides/primitives-examples) to see this coupled with operator transformations.'

In [ ]:
print("ISA ops:", isa_circuit.count_ops())

NameError: name 'isa_circuit' is not defined

## Inspect a few predictions

In [ ]:
points, tours, exact = next(iter(val_loader))
points_device = points.to(device)
pred_tours = model.greedy_decode(points_device).cpu().numpy()

for i in range(min(3, len(pred_tours))):
    pts = points[i].numpy()
    pred = pred_tours[i].tolist()
    label = tours[i].tolist()

    print(f"Instance {i} | exact_label={bool(exact[i].item())}")
    print("  pred_tour :", pred, "len=", round(tour_length(pts, pred), 4))
    print("  label_tour:", label, "len=", round(tour_length(pts, label), 4))
    print()

Instance 0 | exact_label=True
  pred_tour : [0, 1, 2] len= 1.4311
  label_tour: [0, 1, 2] len= 1.4311

Instance 1 | exact_label=True
  pred_tour : [0, 1, 2] len= 1.8547
  label_tour: [0, 1, 2] len= 1.8547

Instance 2 | exact_label=True
  pred_tour : [0, 1, 2] len= 1.526
  label_tour: [0, 1, 2] len= 1.526



## Notes

- Increase `CONFIG["num_cities"]` to make the TSP harder.
- For `num_cities <= exact_threshold`, labels are exact but data generation becomes expensive.
- For larger problems, labels are heuristic, which makes training scalable but less strictly optimal.
- This is a **hybrid quantum machine learning prototype**, meant for experimentation and teaching rather than state-of-the-art TSP performance.